***

* [总目录](../0_Introduction/0_introduction.ipynb)
* [术语表](../0_Introduction/1_glossary.ipynb)
* [第 5 章：成像](5_0_introduction.ipynb)
    * 上一节：[5.4 脏图像与可见度权重](5_4_imaging_weights.ipynb)
    * 下一节：[5.6 成像参数选择的物理依据](5_6_imaging_parameter_decisions.ipynb)

***


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import HTML
import track_simulator
%matplotlib inline
HTML('../style/course.css') #apply general CSS


导入本节所需的专用模块。


In [ ]:
HTML('../style/code_toggle.html')


## 5.5 小角近似的失效与 $w$ 项<a id='imaging:sec:widefield'></a>

前面几节把成像建立在二维 Fourier 关系之上：$V(u,v)$ 是表观天空 $I(l,m)$ 的 Fourier 变换，脏图像是带权采样可见度的反变换。这个关系在窄视场中非常有用，但它不是最完整的测量方程。只要视场变宽，或者阵列的投影基线具有明显 $w$ 分量，天球曲率就会重新进入相位项，使简单二维 FFT 成像失效。

从第 4.6 节的精确表达式出发，标量、窄带、相位跟踪后的可见度应写为

$$
V(u,v,w)=\int\!\!\int \frac{A(l,m)I(l,m)}{n}
\exp\{-2\pi i[ul+vm+w(n-1)]\}\,dl\,dm,
\qquad
n=\sqrt{1-l^2-m^2}.
$$

二维 Fourier 近似等价于令 $n\simeq1$，并忽略 $w(n-1)$。本节讨论的全部宽场效应，核心都来自这个被忽略的相位项。


In [ ]:
FIG_DIR = Path('figures')
FIG_DIR.mkdir(exist_ok=True)
C = 299792458.0
WAVELENGTH_1GHZ = C / 1.0e9


def normalize(arr):
    arr = np.asarray(arr, dtype=float)
    maxval = np.max(np.abs(arr))
    return arr if maxval == 0 else arr / maxval


### 5.5.1 $w$ 项的几何含义

方向余弦 $(l,m,n)$ 描述的是单位天球上的方向，而图像平面只保存 $(l,m)$。当源靠近相位中心时，天球可以近似为切平面；当源远离相位中心时，真实球面与切平面之间的高度差为

$$
n-1=\sqrt{1-l^2-m^2}-1\simeq-\frac{l^2+m^2}{2}.
$$

因此，忽略 $w$ 项带来的相位误差近似为

$$
\Delta\phi_w(l,m)=-2\pi w(n-1)
\simeq \pi w(l^2+m^2).
$$

这个式子给出一个非常实用的尺度判断。若视场半径为 $\theta$，则相位误差量级为 $\pi |w|\theta^2$。宽视场、长基线、低仰角观测和非共面阵列都会增大这个量。当它接近或超过 1 rad 时，简单二维 FFT 图像中的离轴源就会开始展宽、拖尾或出现方向相关旁瓣。


In [ ]:
l = np.linspace(-0.30, 0.30, 401)
m = np.linspace(-0.30, 0.30, 401)
ll, mm = np.meshgrid(l, m)
inside = ll**2 + mm**2 < 0.30**2
n_term = np.sqrt(np.clip(1.0 - ll**2 - mm**2, 1e-10, None))
w_demo = 120.0
phase_error = -2.0 * np.pi * w_demo * (n_term - 1.0)
phase_error[~inside] = np.nan

field_deg = np.linspace(0.0, 12.0, 400)
theta = np.deg2rad(field_deg / 2.0)
w_values = [50, 200, 1000, 5000]

fig, axes = plt.subplots(1, 2, figsize=(12.5, 5.0))
im = axes[0].imshow(phase_error, origin='lower', extent=[l[0], l[-1], m[0], m[-1]], cmap='magma')
axes[0].set_title(r'w-term phase, $w=120$')
axes[0].set_xlabel('l')
axes[0].set_ylabel('m')
plt.colorbar(im, ax=axes[0], label='phase error [rad]')

for w in w_values:
    axes[1].plot(field_deg, np.pi * w * theta**2, lw=2, label=fr'$|w|={w}$')
axes[1].axhline(1.0, color='0.25', lw=1.2, ls='--')
axes[1].set_xlabel('field diameter [deg]')
axes[1].set_ylabel(r'edge phase error $\pi |w|\theta^2$ [rad]')
axes[1].set_title('when the 2D approximation breaks down')
axes[1].grid(alpha=0.25)
axes[1].legend(frameon=False, fontsize=8)
fig.tight_layout()
fig.savefig(FIG_DIR / 'w_term_phase_error_scale.png', dpi=180, bbox_inches='tight')
plt.close(fig)


![w 项相位误差尺度](figures/w_term_phase_error_scale.png)

**图 5.5.1** 左图显示 $w(n-1)$ 在像平面中形成的方向相关相位屏；右图给出不同 $|w|$ 下，视场边缘相位误差随视场直径的增长。相位误差随视场半径平方增长，因此宽场成像很快会离开二维 Fourier 近似的适用范围。


### 5.5.2 共面与非共面采样

若所有测量点都落在同一个 `uvw` 平面上，则可以通过坐标变换把问题重新化为二维 Fourier 关系。纯东西向阵列在某些理想条件下接近这种情况。二维阵列虽然能提供更丰富的瞬时 `uv` 覆盖，但随着地球自转，不同基线通常会扫过不同的 $w$ 值，整个观测集合不再共面。

这带来一个重要事实：非共面基线效应不是单条基线的“小延迟错误”，而是整个视场中的方向相关相位误差。对相位中心做一次延迟补偿只能让中心方向正确，无法同时让整个宽视场都满足 $w=0$ 的二维关系。


In [ ]:
enu_ew = np.array([[-60.0, 0.0, 0.0], [-20.0, 0.0, 0.0], [20.0, 0.0, 0.0], [60.0, 0.0, 0.0]])
enu_2d = np.array([[-45.0, -35.0, 0.0], [55.0, -20.0, 0.0], [10.0, 45.0, 0.0], [-15.0, 8.0, 0.0]])
latitude = 30.0
declination = 35.0
obs_hours = 8.0
int_hours = 2.0 / 60.0
uvw_ew = track_simulator.sim_uv(0.0, declination, obs_hours, int_hours, enu_ew, latitude, False) / WAVELENGTH_1GHZ
uvw_2d = track_simulator.sim_uv(0.0, declination, obs_hours, int_hours, enu_2d, latitude, False) / WAVELENGTH_1GHZ

fig = plt.figure(figsize=(13.0, 5.5))
ax = fig.add_subplot(1, 2, 1, projection='3d')
ax.plot(uvw_ew[:, 0], uvw_ew[:, 1], uvw_ew[:, 2], '.', ms=2, color='#1d3557')
ax.set_title('east-west array')
ax.set_xlabel('u')
ax.set_ylabel('v')
ax.set_zlabel('w')
ax.view_init(elev=18, azim=130)

ax = fig.add_subplot(1, 2, 2, projection='3d')
ax.plot(uvw_2d[:, 0], uvw_2d[:, 1], uvw_2d[:, 2], '.', ms=2, color='#d62828')
ax.set_title('two-dimensional array')
ax.set_xlabel('u')
ax.set_ylabel('v')
ax.set_zlabel('w')
ax.view_init(elev=18, azim=130)
fig.tight_layout()
fig.savefig(FIG_DIR / 'coplanar_vs_non_coplanar_uvw.png', dpi=180, bbox_inches='tight')
plt.close(fig)

fig, axes = plt.subplots(1, 2, figsize=(12.0, 5.0))
for ax, uvw, title, color in [
    (axes[0], uvw_ew, 'east-west array', '#1d3557'),
    (axes[1], uvw_2d, 'two-dimensional array', '#d62828'),
]:
    ax.scatter(uvw[:, 0], uvw[:, 1], c=uvw[:, 2], s=6, cmap='coolwarm')
    ax.set_title(title)
    ax.set_xlabel('u')
    ax.set_ylabel('v')
    ax.set_aspect('equal')
    ax.grid(alpha=0.2)
fig.tight_layout()
fig.savefig(FIG_DIR / 'w_colored_uv_tracks.png', dpi=180, bbox_inches='tight')
plt.close(fig)


![共面与非共面 uvw 采样](figures/coplanar_vs_non_coplanar_uvw.png)

**图 5.5.2** 理想化东西向阵列与二维阵列在 `uvw` 空间中的轨迹。二维阵列给出更丰富的 `uv` 覆盖，但不同测量点的 $w$ 值变化更明显，宽场成像时必须处理非共面项。

![用颜色显示 w 的 uv 轨迹](figures/w_colored_uv_tracks.png)

**图 5.5.3** 同一组轨迹投影到 `uv` 平面后，用颜色标出 $w$ 值。若仅看 `uv` 投影，某些采样点似乎可以直接放入二维 FFT；但颜色显示它们并不对应同一个 $w=0$ 测量平面。


### 5.5.3 忽略 $w$ 项会怎样破坏图像

忽略 $w$ 项时，相位中心附近的结构仍可能看起来正确，因为那里的 $l^2+m^2$ 很小；越远离相位中心，误差越快增大。宽场图像中常见的离轴源拖尾、椭圆状展宽和方向相关旁瓣，都可以从这个二次增长的相位屏理解。

下面的数值实验只保留几何宽场效应，不引入稀疏采样、噪声和去卷积误差。这样可以更干净地看到：若不同 $w$ 层的可见度都被当成 $w=0$ 来成像，离轴结构会被系统性扭曲；若在每个 $w$ 层显式乘回相位屏，结构便能恢复。


In [ ]:
field_size_deg = 10.0
n_pix = 128
axis_deg = np.linspace(-field_size_deg / 2, field_size_deg / 2, n_pix)
axis_rad = np.deg2rad(axis_deg)
ll, mm = np.meshgrid(axis_rad, axis_rad)
inside = ll**2 + mm**2 < 0.95**2
n_dir = np.sqrt(np.clip(1.0 - ll**2 - mm**2, 1e-10, None))

def gaussian_source(l0_deg, m0_deg, sigma_deg, amp):
    l0 = np.deg2rad(l0_deg)
    m0 = np.deg2rad(m0_deg)
    sigma = np.deg2rad(sigma_deg)
    return amp * np.exp(-((ll - l0) ** 2 + (mm - m0) ** 2) / (2.0 * sigma ** 2))

sky = (
    gaussian_source(0.0, 0.0, 0.25, 1.00)
    + gaussian_source(2.8, -2.2, 0.18, 0.75)
    + gaussian_source(-3.4, 2.6, 0.20, 0.45)
    + gaussian_source(1.5, 3.2, 0.12, 0.30)
)
sky[~inside] = 0.0
sky_over_n = sky / n_dir
w_planes = np.array([-1800.0, -1200.0, -600.0, 0.0, 600.0, 1200.0, 1800.0])

image_planes = []
corrected_planes = []
for w_value in w_planes:
    image_with_w = sky_over_n * np.exp(-2j * np.pi * w_value * (n_dir - 1.0))
    vis = np.fft.fftshift(np.fft.fft2(np.fft.ifftshift(image_with_w)))
    wrong_image = np.fft.fftshift(np.fft.ifft2(np.fft.ifftshift(vis)))
    image_planes.append(wrong_image)
    corrected_planes.append(wrong_image * np.exp(2j * np.pi * w_value * (n_dir - 1.0)) * n_dir)

uncorrected = np.real(np.mean(image_planes, axis=0))
corrected = np.real(np.mean(corrected_planes, axis=0))
residual = corrected - sky
extent = [axis_deg[0], axis_deg[-1], axis_deg[0], axis_deg[-1]]

fig, axes = plt.subplots(1, 4, figsize=(16.0, 4.5))
axes[0].imshow(sky, origin='lower', extent=extent, cmap='viridis')
axes[0].set_title('model sky')
axes[1].imshow(np.angle(np.exp(-2j * np.pi * 1200.0 * (n_dir - 1.0))), origin='lower', extent=extent, cmap='twilight')
axes[1].set_title('phase screen')
axes[2].imshow(uncorrected, origin='lower', extent=extent, cmap='viridis')
axes[2].set_title('formed as if w=0')
axes[3].imshow(corrected, origin='lower', extent=extent, cmap='viridis')
axes[3].set_title('after w correction')
for ax in axes:
    ax.set_xlabel('l (deg)')
    ax.set_ylabel('m (deg)')
fig.tight_layout()
fig.savefig(FIG_DIR / 'w_term_image_distortion_demo.png', dpi=180, bbox_inches='tight')
plt.close(fig)


![w 项造成的离轴畸变](figures/w_term_image_distortion_demo.png)

**图 5.5.4** 一个只包含几何 $w$ 项的宽场成像实验。相位中心附近的源受影响较小，远离中心的源在忽略 $w$ 项时明显展宽和变形；显式乘回相位屏后，模型结构基本恢复。


### 5.5.4 常见宽场校正方法

处理 $w$ 项的方法很多，它们的共同目标都是避免把所有可见度错误地压到同一个 $w=0$ 平面上。不同方法在计算位置上有所不同：有的在图像域修正相位，有的在 `uv` 域修改卷积核，有的把天空切成许多小平面。

**快照成像。** 若观测被切成足够短的时间段，每个时间段内的 `uvw` 采样可以近似共面。每张快照先独立成像，再通过坐标重投影合成为总图。该方法概念简单，适合某些低频宽场巡天，但快照之间的重投影和波束变化需要仔细处理。

**多面体成像。** 把大视场划分为许多小 facet，每个 facet 使用自己的切平面。这样每个小区域内 $l^2+m^2$ 较小，二维近似重新变好。若只移动相位中心而不旋转 facet 几何，离原始中心越远的 facet 需要越小；若每个 facet 都与天球局部相切，则可以更有效地控制投影误差。

**W-projection。** 将相位因子

$$
\mathcal{w}_w(l,m)=\exp[-2\pi i w(n-1)]
$$

看成图像域乘法。根据卷积定理，它在 `uv` 域中对应一个卷积核 $\mathcal{W}_w(u,v)$。因此，网格化时可针对不同 $w$ 值选择不同的卷积核，把非共面可见度投影回统一的成像平面。其代价是卷积核支持宽度随视场和 $|w|$ 增大而增加。

**W-stacking。** 把可见度按 $w$ 值分到若干层，每层独立网格化并做二维 FFT，然后在图像域乘以对应的 $w$ 相位校正，最后把各层合成。它把一部分复杂度从 `uv` 卷积核转移到多层图像域相位修正上，常与 W-projection 组合使用。

**A-projection 与 AW-projection。** 真实宽场成像还会受到主波束、馈源偏振、电离层等方向相关效应影响。A-projection 把方向相关增益纳入卷积核，AW-projection 则同时处理 A 项和 W 项。对高动态范围宽场图像，这类方向相关校正往往和 $w$ 项一样重要。


In [ ]:
field_diameter_deg = np.linspace(1.0, 15.0, 200)
field_diameter_rad = np.deg2rad(field_diameter_deg)
wmax_values = [500, 2000, 8000]
quality = 0.1

fig, axes = plt.subplots(1, 2, figsize=(12.0, 4.8))
for wmax in wmax_values:
    theta = field_diameter_rad / 2.0
    n_edge = np.sqrt(np.clip(1.0 - theta**2, 1e-8, None))
    n_planes = 2.0 * np.pi * wmax * np.abs(n_edge - 1.0) / quality
    support = 4.0 * np.pi * wmax * field_diameter_rad**2 / np.sqrt(np.clip(2.0 - field_diameter_rad**2, 1e-8, None))
    axes[0].plot(field_diameter_deg, n_planes, lw=2, label=fr'$w_{{max}}={wmax}$')
    axes[1].plot(field_diameter_deg, support, lw=2, label=fr'$w_{{max}}={wmax}$')

axes[0].set_yscale('log')
axes[0].set_xlabel('field diameter [deg]')
axes[0].set_ylabel('approximate w planes')
axes[0].set_title('w-stacking cost scale')
axes[0].grid(alpha=0.25, which='both')
axes[0].legend(frameon=False, fontsize=8)

axes[1].set_yscale('log')
axes[1].set_xlabel('field diameter [deg]')
axes[1].set_ylabel('relative convolution support')
axes[1].set_title('w-projection kernel scale')
axes[1].grid(alpha=0.25, which='both')
axes[1].legend(frameon=False, fontsize=8)
fig.tight_layout()
fig.savefig(FIG_DIR / 'widefield_correction_cost_scales.png', dpi=180, bbox_inches='tight')
plt.close(fig)


![宽场校正计算量尺度](figures/widefield_correction_cost_scales.png)

**图 5.5.5** 宽场校正的近似计算量尺度。无论采用 W-stacking 还是 W-projection，代价都会随视场和最大 $|w|$ 快速增加。这也是宽场成像通常比窄场成像更昂贵的根本原因。


### 5.5.5 小结

二维 FFT 成像的前提是视场足够小，或者 $w$ 项可以忽略。更一般的宽场成像必须面对天球曲率和非共面基线共同带来的方向相关相位误差。误差尺度近似为 $\pi|w|\theta^2$，因此视场越宽、基线越长，$w$ 项越快变成主导误差源。

这并不意味着 Fourier 成像思想失效，而是说明需要把被省略的几何项重新放回计算中。快照成像、多面体成像、W-projection、W-stacking 以及 AW-projection，都是在不同计算域中处理同一个相位因子 $\exp[-2\pi i w(n-1)]$。第五章到这里完成了从可见度采样到实际成像方程的主要路径；下一章将处理脏图像之后的逆问题，也就是去卷积。


***

* 下一节：[5.6 成像参数选择的物理依据](5_6_imaging_parameter_decisions.ipynb)
